In [23]:
!pip install requests beautifulsoup4 pandas

#Cấu hình
- Khai báo URL của website chứa dữ liệu lịch sử cổ phiếu FPT.
- Thiết lập khoảng thời gian thu thập 5 năm (09/10/2021 – 09/10/2026).
- Giới hạn số trang cần thu thập (MAX_PAGES = 40) và thời gian nghỉ giữa các request (DELAY = 2).
- Tạo requests.Session() để quản lý các HTTP request.

In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime, date


BASE_URL = "https://lichsugia.site/lich-su-giao-dich/FPT.html"

START_DATE = date(2021, 10, 9)
END_DATE = date(2026, 10, 9)

MAX_PAGES = 40
DELAY = 2

session = requests.Session()
session.headers.update({
    "User-Agent": "FPT-Student-Research/1.0"
})




#Chuyển đổi dữ liệu
- parse_price(): Chuyển giá cổ phiếu từ chuỗi thành số thực (float).
- parse_volume(): Loại bỏ ký tự phân cách và chuyển khối lượng giao dịch thành số nguyên (int).
- parse_percent(): Sử dụng Regex để trích xuất phần trăm biến động giá.

In [27]:


def parse_price(value):
    value = value.strip().replace(" ", "")
    value = value.replace(",", ".")
    return float(value)


def parse_volume(value):
    value = re.sub(r"[^\d]", "", value)
    return int(value)


def parse_percent(value):
    match = re.search(
        r"\(([+-]?\d+(?:[.,]\d+)?)\s*%\)",
        value
    )

    if match:
        return float(match.group(1).replace(",", "."))

    return None


#Thu thập dữ liệu
-  Tải nội dung HTML của từng trang, dùng BeautifulSoup để tìm bảng lịch sử giao dịch.
- Lưu thông tin từng phiên giao dịch dưới dạng dictionary và trả về danh sách dữ liệu.


Ý nghĩa của các biến
- symbol:	Mã chứng khoán, ở đây là FPT.
- date:	Ngày diễn ra phiên giao dịch.
- open	Giá mở cửa – giá giao dịch đầu phiên.
- high	Giá cao nhất trong phiên giao dịch.
- low	Giá thấp nhất trong phiên giao dịch.
- close	Giá đóng cửa – giá cuối phiên giao dịch.
- adjusted_close	Giá đóng cửa đã được điều chỉnh theo các sự kiện như chia cổ tức, chia tách cổ phiếu.
- volume	Tổng khối lượng cổ phiếu được giao dịch trong phiên.
- change_percent	Tỷ lệ phần trăm thay đổi giá cổ phiếu so với phiên trước, theo cách tính của nguồn dữ liệu.

In [28]:


def crawl_page(page):

    url = f"{BASE_URL}?page={page}"

    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

    except requests.RequestException as e:
        print("Loi truy cap:", e)
        return None

    soup = BeautifulSoup(response.text, "html.parser")


    table = None

    for t in soup.find_all("table"):
        headings = [
            th.get_text(" ", strip=True)
            for th in t.find_all("th")
        ]

        if (
            "Ngày" in headings
            and "Khối lượng" in headings
            and "Mở cửa" in headings
        ):
            table = t
            break

    if table is None:
        print("Khong tim thay bang tai trang:", page)
        return None

    data = []

    for row in table.find_all("tr"):

        cells = row.find_all("td")

        if len(cells) != 9:
            continue

        values = [
            cell.get_text(" ", strip=True)
            for cell in cells
        ]

        try:
            trading_date = datetime.strptime(
                values[0], "%d/%m/%Y"
            ).date()

            record = {
                "symbol": "FPT",
                "date": trading_date,
                "open": parse_price(values[6]),
                "high": parse_price(values[7]),
                "low": parse_price(values[8]),
                "close": parse_price(values[2]),
                "adjusted_close": parse_price(values[1]),
                "volume": parse_volume(values[4]),
                "change_percent": parse_percent(values[3])
            }

            data.append(record)

        except (ValueError, IndexError):
            continue

    return data



all_data = []
reached_start = False
seen_pages = set()

for page in range(1, MAX_PAGES + 1):

    print(f"Dang thu thap trang {page}...")

    page_data = crawl_page(page)

    if page_data is None:
        print("Dung do loi truy cap hoac HTML.")
        break

    if len(page_data) == 0:
        print("Khong con du lieu.")
        break

    page_signature = (
        max(item["date"] for item in page_data),
        min(item["date"] for item in page_data)
    )

    if page_signature in seen_pages:
        print("Trang du lieu bi lap, dung lai.")
        break

    seen_pages.add(page_signature)

    oldest_date = min(item["date"] for item in page_data)

    # Chi giu du lieu trong khoang 5 nam
    valid_data = [
        item for item in page_data
        if START_DATE <= item["date"] <= END_DATE
    ]

    all_data.extend(valid_data)

    print("So ban ghi hop le:", len(valid_data))
    print("Ngay cu nhat trang:", oldest_date)

    # Dung khi da den moc 5 nam
    if oldest_date <= START_DATE:
        reached_start = True
        print("Da den moc 5 nam.")
        break

    time.sleep(DELAY)

Dang thu thap trang 1...
So ban ghi hop le: 50
Ngay cu nhat trang: 2026-07-29
Dang thu thap trang 2...
So ban ghi hop le: 50
Ngay cu nhat trang: 2026-05-20
Dang thu thap trang 3...
So ban ghi hop le: 50
Ngay cu nhat trang: 2026-03-05
Dang thu thap trang 4...
So ban ghi hop le: 50
Ngay cu nhat trang: 2025-12-16
Dang thu thap trang 5...
So ban ghi hop le: 50
Ngay cu nhat trang: 2025-10-07
Dang thu thap trang 6...
So ban ghi hop le: 50
Ngay cu nhat trang: 2025-07-25
Dang thu thap trang 7...
So ban ghi hop le: 50
Ngay cu nhat trang: 2025-05-16
Dang thu thap trang 8...
So ban ghi hop le: 50
Ngay cu nhat trang: 2025-03-03
Dang thu thap trang 9...
So ban ghi hop le: 50
Ngay cu nhat trang: 2024-12-12
Dang thu thap trang 10...
So ban ghi hop le: 50
Ngay cu nhat trang: 2024-10-03
Dang thu thap trang 11...
So ban ghi hop le: 50
Ngay cu nhat trang: 2024-07-23
Dang thu thap trang 12...
So ban ghi hop le: 50
Ngay cu nhat trang: 2024-05-14
Dang thu thap trang 13...
So ban ghi hop le: 50
Ngay cu nhat 

#Tạo và làm sạch DataFrame
- Chuyển danh sách bản ghi thành bảng dữ liệu, loại bỏ các bản ghi trùng ngày giao dịch và lưu dữ liệu
- Hiển thị 10 bản ghi đầu và 10 bản ghi cuối.

In [29]:

df = pd.DataFrame(all_data)

if not df.empty:

    df = df.drop_duplicates(
        subset=["symbol", "date"]
    )

    df = df.sort_values("date").reset_index(drop=True)

    df["date"] = pd.to_datetime(df["date"])
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")



    df.to_csv(
        "FPT_stock.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df.to_csv(
        "FPT_stock.tsv",
        sep="\t",
        index=False
    )

    df.to_json(
        "FPT_stock.json",
        orient="records",
        force_ascii=False,
        indent=4
    )

    # =========================================
    # HIEN THI KET QUA
    # =========================================

    print("\n===== KET QUA THU THAP =====")

    print("Ma chung khoan: FPT")
    print("Tong so ban ghi:", len(df))
    print("Ngay bat dau:", df["date"].min())
    print("Ngay ket thuc:", df["date"].max())

    if not reached_start:
        print("CANH BAO: Chua xac nhan du 5 nam du lieu.")

    print("\nThong ke du lieu:")
    print(df.describe())

    print("\n10 dong dau tien:")
    display(df.head(10))

    print("\n10 dong cuoi cung:")
    display(df.tail(10))

    print("\nDONE.")

else:
    print("ERROR: Khong thu thap duoc du lieu.")



===== KET QUA THU THAP =====
Ma chung khoan: FPT
Tong so ban ghi: 1244
Ngay bat dau: 2021-10-11
Ngay ket thuc: 2026-10-09

Thong ke du lieu:
              open        high          low        close  adjusted_close  \
count  1244.000000  1244.00000  1244.000000  1244.000000     1244.000000   
mean     99.399196   100.49590    98.314630    99.358039       73.333545   
std      22.078899    22.26935    21.910629    22.118928       23.403116   
min      59.500000    60.00000    57.500000    57.900000       40.500000   
25%      81.600000    82.07500    81.000000    81.600000       49.600000   
50%      95.500000    96.40000    94.650000    95.500000       70.800000   
75%     114.000000   115.80000   112.600000   114.075000       94.685000   
max     154.100000   156.00000   153.400000   154.300000      130.760000   

             volume  change_percent  
count  1.244000e+03     1244.000000  
mean   4.459503e+06        0.044413  
std    4.486118e+06        1.763179  
min    2.210000e+05  

,symbol,date,open,high,low,close,adjusted_close,volume,change_percent
0,FPT,2021-10-11,99.0,99.3,98.3,99.2,49.82,2295500,0.30
1,FPT,2021-10-12,99.2,99.2,98.5,99.0,49.72,1125300,-0.20
2,FPT,2021-10-13,99.0,101.5,98.5,99.4,49.92,3232500,0.40
3,FPT,2021-10-14,99.4,100.0,98.4,98.8,49.62,1824200,-0.60
4,FPT,2021-10-15,99.4,100.1,98.4,98.9,49.67,1702500,0.10
5,FPT,2021-10-18,99.0,100.0,98.6,99.0,49.72,2264100,0.10
6,FPT,2021-10-19,99.0,99.5,98.6,98.6,49.52,1534300,-0.40
7,FPT,2021-10-20,98.7,99.5,96.6,97.6,49.01,2248600,-1.01
8,FPT,2021-10-21,98.4,98.5,96.7,96.7,48.56,2283600,-0.92
9,FPT,2021-10-22,97.0,97.4,96.1,96.2,48.31,2023400,-0.52



10 dong cuoi cung:


,symbol,date,open,high,low,close,adjusted_close,volume,change_percent
1234,FPT,2026-09-28,64.7,64.8,63.0,63.7,63.7,5148700,-1.55
1235,FPT,2026-09-29,63.7,64.1,63.1,63.2,63.2,3118000,-0.78
1236,FPT,2026-09-30,63.4,64.0,63.0,63.0,63.0,3041400,-0.32
1237,FPT,2026-10-01,63.0,63.1,62.6,62.7,62.7,3413000,-0.48
1238,FPT,2026-10-02,62.7,63.2,62.1,62.1,62.1,3199600,-0.96
1239,FPT,2026-10-05,62.2,62.8,61.9,61.9,61.9,2513400,-0.32
1240,FPT,2026-10-06,62.0,62.2,60.0,60.4,60.4,8214700,-2.42
1241,FPT,2026-10-07,60.4,61.3,59.7,59.7,59.7,5438300,-1.16
1242,FPT,2026-10-08,59.5,61.5,58.9,59.7,59.7,6574800,0.00
1243,FPT,2026-10-09,59.7,60.0,57.5,57.9,57.9,10851700,-3.02



DONE.
